# Leçon 01 - Introduction aux Agents IA

Bienvenue dans la première leçon du cours **Agents IA pour débutants** !

Un **agent IA** est un programme qui utilise un grand modèle de langage (LLM) comme moteur de raisonnement et peut prendre des *actions* dans le monde réel — appeler des API, interroger des bases de données ou exécuter du code — pour accomplir un objectif au nom d'un utilisateur.

Dans ce carnet, vous allez construire votre premier agent : un **agent de voyage** qui recommande des destinations de vacances. En cours de route, vous apprendrez à :

1. Vous connecter au service Azure AI Foundry Agent en utilisant le **Microsoft Agent Framework**.
2. Donner à l'agent un **outil** — une fonction Python simple qu'il peut appeler.
3. Exécuter l'agent et inspecter sa réponse.
4. Diffuser la réponse de l'agent token par token.


## Configuration

Avant d'exécuter ce notebook, assurez-vous d'avoir :

1. **Un projet Azure AI Foundry** avec un modèle de chat déployé (par exemple `gpt-4o-mini`).
2. **Connecté avec l'interface CLI Azure** — exécutez `az login` dans votre terminal.
3. **Défini les variables d'environnement requises :**
   - `AZURE_AI_PROJECT_ENDPOINT` — le point de terminaison de votre projet Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — le nom de votre modèle déployé.

La cellule ci-dessous installe les paquets Python dont vous avez besoin.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Créer votre premier agent

Un agent a besoin de deux choses :

- **Des instructions** qui lui disent *qui il est* et *comment se comporter* (un prompt système).
- **Des outils** — des fonctions Python décorées avec `@tool` que l'agent peut appeler pour récupérer des informations ou effectuer des actions.

Ci-dessous, nous définissons un outil simple qui renvoie une liste de destinations de vacances populaires. L'agent utilisera cet outil lorsqu'un utilisateur demandera des recommandations de voyage.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Réponses en streaming

Pour une expérience plus interactive, vous pouvez **diffuser** la réponse de l'agent. Au lieu d'attendre la réponse complète, l'agent fournit des fragments de texte au fur et à mesure de leur génération. Cela est particulièrement utile dans les interfaces de chat où vous souhaitez afficher les résultats en temps réel.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Résumé

Dans cette leçon, vous avez appris à :

- **Créer un fournisseur** qui se connecte au service Azure AI Foundry Agent via `AzureAIProjectAgentProvider`.
- **Définir un outil** en utilisant le décorateur `@tool` afin que l'agent puisse appeler vos fonctions Python.
- **Exécuter l'agent** avec un message utilisateur et afficher sa réponse.
- **Diffuser les réponses** pour une sortie en temps réel.

Dans la leçon suivante, nous explorerons plus en profondeur les cadres agentiques et apprendrons comment doter les agents d'outils plus puissants et de capacités de raisonnement en plusieurs étapes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
